# 02 · Publicación de datos para Looker Studio
**EP5 · UNA SOLA tabla CSV de producción**

Se elimina el patrón de múltiples CSV (`generos_long.csv`, `paises_long.csv`, etc.). El tablero utilizará exclusivamente `data/processed/catalogo_looker_ep5.csv`, **una fila por título único**.

**Tradeoff declarado:** el filtro de género usa `genero_principal`, no todos los géneros de cada obra. La lista multigénero completa se guarda como campo informativo. Si en el futuro necesitan análisis multietiqueta preciso, será necesario usar una tabla puente separada o una base relacional; un único CSV plano no puede representar simultáneamente ambos granos sin duplicar títulos.

In [5]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / 'src' / 'pipeline.py').exists() else Path.cwd().parent
assert (ROOT / 'src' / 'pipeline.py').exists(), 'Ejecuta Jupyter desde la raíz del proyecto o notebooks/'
sys.path.insert(0,str(ROOT))
import pandas as pd, numpy as np
from IPython.display import display
from src.pipeline import load_sources, clean, audit, kpis, looker_table, OUT


## 1. Generar desde fuentes originales

In [6]:
from src.pipeline import export
df,looker=export()
print("Exportado:", OUT.relative_to(ROOT))
print("Registros:",len(looker),"Columnas:",len(looker.columns))
display(looker.head(3))

Exportado: data\processed\looker\catalogo_looker_ep5.csv
Registros: 31991 Columnas: 27


,content_key,type,title,anio_estreno,date_added,genero_principal,generos_armonizados,cantidad_generos,pais_principal,idioma_codigo,...,nota_confiable_50,tiene_50_votos,finanzas_validas,presupuesto_usd,ingresos_usd,multiplo_ingresos,roi_neto,ganancia_usd,recupera_presupuesto,rango_presupuesto
0,M_10192,Movie,Shrek Forever After,2010,2010-05-16,Comedy,Comedy | Aventura | Fantasía | Animation | Family,5,United States of America,en,...,6.380,1,True,165000000.0,752600867.0,4.561217,3.561217,587600867.0,True,> 100 M
1,M_27205,Movie,Inception,2010,2010-07-15,Acción,Acción | Ciencia ficción | Aventura,3,United Kingdom,en,...,8.369,1,True,160000000.0,839030630.0,5.243941,4.243941,679030630.0,True,> 100 M
2,M_12444,Movie,Harry Potter and the Deathly Hallows: Part 1,2010,2010-11-17,Aventura,Aventura | Fantasía,2,United Kingdom,en,...,7.744,1,True,250000000.0,954305868.0,3.817223,2.817223,704305868.0,True,> 100 M


## 2. Pruebas de integridad y cobertura

In [7]:
assert looker.content_key.is_unique
assert len(looker)==31991
assert (looker.loc[looker.type=="TV Show","finanzas_validas"]==False).all()
assert looker.loc[looker.finanzas_validas,"presupuesto_usd"].ge(10000).all()
assert looker.loc[looker.finanzas_validas,"ingresos_usd"].ge(10000).all()
assert looker.loc[looker.tiene_50_votos==0,"nota_confiable_50"].isna().all()
print("6 controles de calidad aprobados")
print("Nulos informativos por campo:")
display(looker.isna().sum().sort_values(ascending=False).head(12))

6 controles de calidad aprobados
Nulos informativos por campo:


ganancia_usd            28499
rango_presupuesto       28499
roi_neto                28499
recupera_presupuesto    28499
presupuesto_usd         28499
multiplo_ingresos       28499
ingresos_usd            28499
nota_confiable_50       16594
temporadas              16000
director_principal      11092
nota_con_votos           4560
pais_principal           2261
dtype: int64

## 3. Control independiente de KPIs de publicación

In [8]:
control=pd.Series(kpis(df),name="valor")
display(control.to_frame())
verificacion=pd.read_csv(OUT,encoding="utf-8-sig")
print("Filas después de relectura:",len(verificacion))
print("Promedio popularidad (control, no KPI comparativo):",round(verificacion.popularity.mean(),3))
print("Nota >=50 votos:",round(verificacion.nota_confiable_50.mean(),3))
print("Recupera presupuesto:",round(verificacion.loc[verificacion.finanzas_validas,"recupera_presupuesto"].mean()*100,2),"%")

,valor
titulos_unicos,3.199100e+04
peliculas,1.600000e+04
series,1.599100e+04
n_con_50_votos,1.539700e+04
nota_media_50_votos,6.626000e+00
popularidad_media_total_solo_control,4.262600e+01
peliculas_financieras,3.492000e+03
multiplo_ingresos_agregado,2.924100e+00
multiplo_ingresos_mediano,1.719600e+00
pct_recupera_presupuesto,6.386000e+01


Filas después de relectura: 31991
Promedio popularidad (control, no KPI comparativo): 42.626
Nota >=50 votos: 6.626
Recupera presupuesto: 63.86 %


## 4. Configuración obligatoria de Looker Studio
1. Conectar **solo** `catalogo_looker_ep5.csv`. Reemplazar la fuente anterior; no mezclar con los CSV antiguos.
2. `content_key`: dimensión texto. Para total de contenidos usar **COUNT_DISTINCT(content_key)**.
3. `anio_estreno`: número entero; `date_added`: fecha (no indicador de evolución real de incorporación).
4. `popularity`, `popularidad_percentil_tipo`, `nota_confiable_50`, `multiplo_ingresos`, `presupuesto_usd`, `ingresos_usd`: **número decimal** con separador interpretado correctamente. Comprobar el máximo real de `popularity` (≈ 6.422), no millones.
5. `finanzas_validas`: booleano. En la vista financiera aplicar filtro fijo `finanzas_validas = TRUE`.
6. `recupera_presupuesto`: booleano; calcular `% recupera presupuesto` como promedio del booleano (o equivalente), solamente sobre películas financieras válidas.
7. El valor agregado de múltiplo financiero se calcula con **SUM(ingresos_usd) / SUM(presupuesto_usd)**, nunca con promedio simple de múltiplos individuales.
8. Para gráficos de valoración usar `nota_confiable_50` y excluir valores nulos; mostrar el **n** de títulos elegibles.
9. Para rankings de género o país, filtrar nulos en la dimensión específica; no imponer la exclusión a todo el catálogo.
10. Usar controles comunes de año, tipo, género principal, país principal e idioma. El análisis financiero es exclusivo de Movie.

**Importante:** fechas/años y composición 50/50 resultan del muestreo. No presentar crecimiento del catálogo a partir de esos gráficos.

**Reglas de color propuestas:** Movie = azul `#3366CC`, TV Show = naranja `#E68A2E`; usar la misma codificación en las figuras estáticas.

## 5. Diccionario corto de métricas
**`nota_con_votos`**: media (0–10) con >=1 voto, solo indicador de cobertura. **`nota_confiable_50`**: media con >=50 votos, comparaciones principales. **`popularidad_percentil_tipo`**: posición porcentual dentro de Movie o TV Show (no popularidad absoluta). **`multiplo_ingresos`**: ingresos / presupuesto. **`roi_neto`**: múltiplo - 1. **`recupera_presupuesto`**: bandera ingresos >= presupuesto en finanzas válidas.

## Nuevos hallazgos y límites de la exportación
- Total publicable: **31.991 títulos** (9 conflictos deduplicados; documentado en notebook 01).
- El ranking de géneros principales evita contar varias veces el mismo título, pero **no debe llamarse ranking de presencia total de géneros**.
- Las calificaciones filtradas con >=50 votos permiten comparaciones menos expuestas a notas con 1–9 votos, al precio de reducir cobertura.
- El nuevo CSV conserva nulos como nulos, no los rotula como `Sin dato` en los rankings.
- El tablero no debe mezclar medias de popularidad de las series y películas sin indicar la incompatibilidad de distribuciones.

**Salida única:** `data/processed/catalogo_looker_ep5.csv`. No se generan tablas auxiliares de dimensiones.